# Lezione 10 — Word embedding

**Corso:** NLP Classico con Python · **Blocco 4:** Significato e ricerca
**Tempo stimato:** 110 minuti · **Capitolo del manuale:** `manual/10_word_embeddings_IT.md`

## Cosa vedrai in questa lezione

- L'**ipotesi distribuzionale** («una parola si conosce dalle sue vicine») e il passaggio da vettori **sparsi** (conteggi di contesto) a vettori **densi**, con un esempio che si calcola a mano.
- La **similarità del coseno** tra due vettori, calcolata a mano e con NumPy.
- **Word2Vec** (skip-gram e CBOW, con la finestra come parametro) e **fastText** con `gensim`, su 3-4 opere di pubblico dominio per lingua.
- **Vicini più prossimi** e **analogie** come dimostrazione dei **limiti**, non come risultato: con qualche centinaio di migliaia di parole i vettori sono **rumorosi**, e lo misuriamo con altri semi e altri parametri.
- **fastText** davanti a parole rare, **parole fuori vocabolario** e **refusi**.
- Limiti: **un vettore per parola**, distorsioni del corpus, poco testo, e un confronto con un **corpus sintetico moderno**.

**Requisiti:** lezioni 4 (sacco di parole, vettori di conteggi) e 9 (temi: un altro modo di riassumere il significato); la 11 proseguirà con la ricerca classica. **Librerie:** `gensim`, `numpy`, `pandas`, `matplotlib`.
**Dati:** `datos/gutenberg/embeddings_es_obras.txt` e `datos/gutenberg/embeddings_it_obras.txt` (Quijote, La Regenta, Marianela e Lazarillo; Promessi sposi, Pinocchio, La vita militare e Novelle; provenienza in `datos/LEEME.md`), `datos/resenas_*.csv` e `datos/tickets_*.csv` (in Colab, carica la cartella `datos/`). I testi letterari sono di pubblico dominio e di uno-quattro secoli fa: **non rappresentano lo spagnolo né l'italiano di oggi**. Tutte le cifre sono un *esempio illustrativo*. **Durata dell'esecuzione:** circa due minuti.

---
## Cella 0 — Setup

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("gensim", "gensim")
print("Librerie pronte.")

In [ ]:
import os
import re
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gensim.models import Word2Vec, FastText

warnings.filterwarnings("ignore")
random.seed(42)                          # seme fisso
np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

def frases_de_texto(texto):
    """Minuscole, taglio in frasi e solo lettere; scarta le frasi di meno di 3 parole."""
    salida = []
    for trozo in re.split(r"[.!?;:]+\s|\n+", texto.lower()):
        palabras = re.findall(r"[^\W\d_]+", trozo)
        if len(palabras) >= 3:
            salida.append(palabras)
    return salida

def entrenar_w2v(frases, sg=1, window=5, seed=1, epochs=5, vector_size=50, min_count=5):
    """Word2Vec riproducibile: workers=1 e seme fisso (sg=1 skip-gram, sg=0 CBOW)."""
    return Word2Vec(frases, vector_size=vector_size, window=window, min_count=min_count, sg=sg,
                    epochs=epochs, workers=1, seed=seed)

def entrenar_ft(frases, sg=1, window=5, seed=1, epochs=5, vector_size=50, min_count=5):
    """fastText riproducibile, con n-grammi di caratteri da 3 a 5."""
    return FastText(frases, vector_size=vector_size, window=window, min_count=min_count, sg=sg,
                    epochs=epochs, workers=1, seed=seed, min_n=3, max_n=5)

def coseno(a, b):
    """Similarità del coseno tra due vettori."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def vecinos(modelo, palabra, n=6):
    """Le n parole più vicine per coseno."""
    return [w for w, _ in modelo.wv.most_similar(palabra, topn=n)]

def frecuentes(frases, modelo, n=200, largo=4):
    """Le n parole più frequenti (di almeno `largo` lettere) presenti nel vocabolario del modello."""
    cuenta = Counter(w for f in frases for w in f)
    return [w for w, _ in cuenta.most_common() if len(w) >= largo and w in modelo.wv.key_to_index][:n]

def solape(m1, m2, palabras, k=10):
    """Frazione media di vicini (top k) condivisi da due modelli: 1 = identici, 0 = nessuno."""
    valores = []
    for w in palabras:
        a, b = set(vecinos(m1, w, k)), set(vecinos(m2, w, k))
        valores.append(len(a & b) / k)
    return float(np.mean(valores))

def rango_analogia(modelo, a, b, c, objetivo):
    """a - b + c: (posizione dell'obiettivo tra i candidati, primo candidato). Esclude a, b e c."""
    if objetivo not in modelo.wv.key_to_index:
        return None, None
    sims = modelo.wv.most_similar(positive=[a, c], negative=[b], topn=None)
    excluir = {modelo.wv.key_to_index[w] for w in (a, b, c)}
    candidatos = [i for i in np.argsort(-sims) if i not in excluir]
    return candidatos.index(modelo.wv.key_to_index[objetivo]) + 1, modelo.wv.index_to_key[candidatos[0]]

def frases_modernas(idioma):
    """Frasi del corpus sintetico moderno (recensioni + ticket, colonna texto_limpio)."""
    salida = []
    for nombre in (f"resenas_{idioma}.csv", f"tickets_{idioma}.csv"):
        for texto in pd.read_csv(cargar_datos(nombre))["texto_limpio"]:
            for trozo in re.split(r"[.!?;:]+\s", str(texto).lower()):
                palabras = re.findall(r"[^\W\d_]+", trozo)
                if len(palabras) >= 3:
                    salida.append(palabras)
    return salida

print("Funzioni ausiliarie caricate.")

---
## 1. Similarità del coseno, a mano

Un embedding è un **vettore** per parola. Per dire che due parole «si somigliano» confrontiamo i loro vettori con il **coseno** dell'angolo che formano:

$$\cos(a,b)=\frac{a\cdot b}{\|a\|\,\|b\|}$$

Va da −1 (versi opposti) a 1 (stessa direzione); 0 significa ortogonali. Piccolo esempio (*esempio illustrativo*): `a = (3, 4)` e `b = (4, 3)`. Prima di eseguire: `a·b = 12 + 12 = 24`, `‖a‖ = ‖b‖ = 5`, quindi `cos = 24/25 = 0,96`.

In [ ]:
a, b = np.array([3, 4]), np.array([4, 3])
print("a·b =", int(a @ b), "| ‖a‖ =", np.linalg.norm(a), "| ‖b‖ =", np.linalg.norm(b))
print("cos(a, b) =", coseno(a, b))
print("cos(a, (-4, 3)) =", coseno(a, [-4, 3]), "| cos(a, -a) =", coseno(a, -a))
assert np.isclose(coseno(a, b), 0.96) and np.isclose(coseno(a, [-4, 3]), 0.0) and np.isclose(coseno(a, -a), -1.0)   # coincide con il calcolo a mano

### Cosa osservi?

`cos(a, b) = 0,96`: quasi la stessa direzione. `(−4, 3)` è perpendicolare ad `a` (prodotto scalare 0 → coseno 0) e `−a` punta al contrario (−1). **Il coseno guarda solo la direzione, non la lunghezza**: `(3, 4)` e `(30, 40)` hanno coseno 1. Per questo si usa con gli embedding, dove la lunghezza del vettore dipende soprattutto dalla frequenza della parola.

---
## 2. Ipotesi distribuzionale: dal vettore sparso al denso

*«Conoscerai una parola dalla compagnia che frequenta»* (Firth, 1957). Se due parole compaiono tra le stesse vicine, avranno un significato simile. La versione più semplice: per ogni parola, un vettore con **un conteggio per ogni parola del vocabolario** (quante volte compare proprio accanto a quella parola). È un vettore **sparso**: tante dimensioni quante sono le parole, quasi tutti zeri.

Mini-corpus di 4 frasi, finestra di 1 parola per lato. A mano: i contesti di `gato` sono `el` (2 volte: nella frase 1 e nella 3), `bebe` (1) e `duerme` (1); quelli di `perro` sono esattamente gli stessi, quindi `cos(gato, perro) = 1`. `leche` compare solo accanto a `bebe` (1 volta): `cos(gato, leche) = 1/√6 ≈ 0,408`.

In [ ]:
# datos-es
# Mini-corpus di 4 frasi (finestra di 1 parola per lato)
MINI = [["el", "gato", "bebe", "leche"], ["el", "perro", "bebe", "agua"], ["el", "gato", "duerme"], ["el", "perro", "duerme"]]
VOCAB = ["el", "gato", "perro", "bebe", "duerme", "leche", "agua"]

In [ ]:
indice = {p: i for i, p in enumerate(VOCAB)}
M = np.zeros((len(VOCAB), len(VOCAB)), dtype=int)        # riga = parola, colonna = parola di contesto
for frase in MINI:
    for i, p in enumerate(frase):
        for j in (i - 1, i + 1):                          # finestra di 1 per lato
            if 0 <= j < len(frase):
                M[indice[p], indice[frase[j]]] += 1
print(pd.DataFrame(M, index=VOCAB, columns=VOCAB))
cos_gp = coseno(M[indice["gato"]], M[indice["perro"]])
cos_gl = coseno(M[indice["gato"]], M[indice["leche"]])
print("cos(gato, perro) =", round(cos_gp, 3), "| cos(gato, leche) =", round(cos_gl, 3))
assert M[indice["gato"], 0] == 2 and np.isclose(cos_gp, 1.0) and np.isclose(cos_gl, 1 / np.sqrt(6))   # coincide con il calcolo a mano

### Cosa osservi?

La matrice ha 7 × 7 = 49 caselle per un vocabolario di 7 parole. Con un vocabolario reale di circa 10 000 parole sarebbero **10 000 dimensioni per parola**, quasi tutte zero. `gato` e `perro` risultano identici perché in questo mini-corpus svolgono lo stesso ruolo. Gli **embedding densi** (Word2Vec, fastText) comprimono quell'informazione in qualche decina di dimensioni (qui, 50) che si **imparano** prevedendo parole del contesto; in cambio, le dimensioni non significano più nulla da sole e il risultato dipende dal caso dell'addestramento.

---
## 3. I dati: 3-4 opere per lingua e un corpus moderno minuscolo

- **Spagnolo:** *Don Quijote* (1605-1615), *La Regenta* (1884-1885), *Marianela* (1878) e il *Lazarillo de Tormes* (1554).
- **Italiano:** *I promessi sposi* (1840-1842), *Le avventure di Pinocchio* (1883), *La vita militare* (ed. 1869) e *Novelle* (ed. 1884).
- Tutte di pubblico dominio (Project Gutenberg), unite in un file per lingua. Ogni riga passa in minuscolo, si taglia in frasi (`. ! ? ; :` e a capo) e si tengono solo le **lettere**: cifre e punteggiatura spariscono (e con loro gli apostrofi: `dell'uomo` dà `dell` e `uomo`; una semplificazione che dichiariamo).
- **Corpus moderno sintetico:** il `texto_limpio` delle recensioni e dei ticket del corso (700 testi per lingua). È **minuscolo** (circa 11 000 parole): con questo un modello di embedding non impara quasi nulla. Serve solo a vedere il contrasto di registro.

In [ ]:
# datos-es
ARCHIVOS = {"es": "gutenberg/embeddings_es_obras.txt", "it": "gutenberg/embeddings_it_obras.txt"}
# Parole di ricerca fissate PRIMA di guardare i risultati (non si scelgono i vicini «belli»)
CONSULTAS = {"es": ["rey", "amor", "dinero", "mujer", "muerte", "caballero", "casa", "dios"],
             "it": ["re", "amore", "denaro", "donna", "morte", "cavaliere", "casa", "dio"]}
GENERO = {"es": ("mujer", "hombre"), "it": ("donna", "uomo")}
ANALOGIAS = {"es": [("rey", "hombre", "mujer", "reina"), ("padre", "hombre", "mujer", "madre"), ("señor", "hombre", "mujer", "señora")],
             "it": [("re", "uomo", "donna", "regina"), ("padre", "uomo", "donna", "madre"), ("signore", "uomo", "donna", "signora")]}
POLISEMIA = {"es": ["vino", "cabo"], "it": ["porta", "sale"]}
FUERA_DE_VOCABULARIO = {"es": ["caballeroo", "cabalero", "quijotesco", "hidalguez"],
                        "it": ["cavalliere", "prommesso", "signorotti", "spaventosissimo"]}

In [ ]:
FRASES, MODERNO = {}, {}
for idioma, ruta in ARCHIVOS.items():
    with open(cargar_datos(ruta), encoding="utf-8") as f:
        FRASES[idioma] = frases_de_texto(f.read())
    MODERNO[idioma] = frases_modernas(idioma)
    print(idioma, "| letterario: frasi", len(FRASES[idioma]), "| parole", sum(map(len, FRASES[idioma])),
          "| moderno: frasi", len(MODERNO[idioma]), "| parole", sum(map(len, MODERNO[idioma])))
    assert sum(map(len, FRASES[idioma])) > 400_000 and sum(map(len, MODERNO[idioma])) < 15_000

---
## 4. Word2Vec skip-gram: vicini più prossimi

**Skip-gram** addestra una piccola rete per **prevedere le parole del contesto a partire dalla parola centrale**; i vettori sono i pesi appresi. Parametri che fissiamo (*esempio illustrativo*): `vector_size=50`, `window=5` (fino a 5 parole per lato), `min_count=5` (si ignorano le parole che compaiono meno di 5 volte), `epochs=5`, `workers=1` e un `seed` fisso (con un solo thread il risultato è riproducibile; con più thread, no).

Le 8 parole di ricerca sono **fissate in anticipo** nella cella dei dati: non scegliamo quelle che vengono bene. Guarda i risultati con spirito critico.

> **Nota:** durante l'addestramento può comparire in rosso un messaggio come `Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'`. Non è un errore del tuo codice né influisce sul risultato: è un avviso cosmetico dell'estensione compilata in C di gensim (legato a come libera la memoria in alcune combinazioni di sistema operativo e libreria BLAS). Si può ignorare in sicurezza.

In [ ]:
BASE = {}
for idioma in FRASES:
    BASE[idioma] = entrenar_w2v(FRASES[idioma], sg=1, window=5, seed=1)
    print("=" * 8, idioma, "| vocabolario (min_count=5):", len(BASE[idioma].wv), "| dimensione:", BASE[idioma].wv.vector_size)
    for palabra in CONSULTAS[idioma]:
        vec = BASE[idioma].wv.most_similar(palabra, topn=6)
        print(f"{palabra:>10} ->", ", ".join(f"{w} ({s:.2f})" for w, s in vec))
    assert len(BASE[idioma].wv) > 5000

### Cosa osservi?

Ci sono risultati **ragionevoli** (`caballero` → `andante`, `hidalgo`; `donna` → `signora`, `ragazza`; `casa` → `chiesa`, `camera`) e altri **brutti o arcaici**: `dios` → `vuesa`, `suplico` (forme di cortesia del Seicento), `dinero` → `testamento`, `sabéis`, `dio` → `vossignoria`, `re` → `podestà`, `marchese` (cariche della Milano del Seicento). I vicini sono **parole che compaiono in contesti simili**, non sinonimi: `amor` compare accanto a `desprecio` (opposto, stesso contesto) e `muerte` accanto a `pobreza`. E la similarità **non misura la qualità**: `denaro` → `ohi`, `dirti`, `melampo` con similarità 0,98 (spazzatura), mentre `caballero` → `andante` esce con 0,90. Le cifre dipendono dal corpus e dal seme: **non sono «probabilità di azzeccare»**.

---
## 5. Quanto di tutto questo è rumore? Altri semi, CBOW e un'altra finestra

Addestriamo di nuovo cambiando **una sola cosa** alla volta e misuriamo quanti dei 10 vicini più prossimi si conservano (`solape`: 1 = identici, 0 = nessuno), sulle 8 parole di ricerca e sulle 200 parole più frequenti (di almeno 4 lettere).

- **Un altro seme** (`seed=2`): stesso algoritmo, stessi dati.
- **CBOW** (`sg=0`): prevede la parola centrale a partire dal contesto; è più veloce e di solito va bene con le parole frequenti.
- **Finestra 2** e **finestra 8**: una finestra corta coglie relazioni più sintattiche (parole intercambiabili), una lunga più tematiche (parole dello stesso argomento).

In [ ]:
VARIANTES = {"seed 2": dict(sg=1, window=5, seed=2), "CBOW": dict(sg=0, window=5, seed=1),
             "window 2": dict(sg=1, window=2, seed=1), "window 8": dict(sg=1, window=8, seed=1)}
OTROS = {idioma: {n: entrenar_w2v(FRASES[idioma], **p) for n, p in VARIANTES.items()} for idioma in FRASES}
tabla = []
for idioma in FRASES:
    frec = frecuentes(FRASES[idioma], BASE[idioma])
    for nombre, modelo in OTROS[idioma].items():
        tabla.append([idioma, nombre, round(solape(BASE[idioma], modelo, CONSULTAS[idioma]), 2), round(solape(BASE[idioma], modelo, frec), 2)])
res = pd.DataFrame(tabla, columns=["lingua", "variante", "sovrapposizione (8 ricerche)", "sovrapposizione (200 frequenti)"])
print(res.to_string(index=False))
res.pivot(index="variante", columns="lingua", values="sovrapposizione (200 frequenti)").plot.bar(rot=0)
plt.ylim(0, 1); plt.ylabel("frazione dei 10 vicini che si conserva")
plt.title("Sovrapposizione con il modello di base (skip-gram, finestra 5, seme 1)")
plt.show()
assert (res.iloc[:, 3] < 1).all() and (res.iloc[:, 3] > 0).all()

### Cosa osservi?

Già solo cambiando il seme si perde circa il **40 %** dei vicini sulle 200 parole frequenti (sovrapposizione ≈ 0,58) e ancora di più sulle 8 di ricerca (0,50 in spagnolo e 0,38 in italiano; essendo poche, la cifra è più irregolare). Cambiare la finestra peggiora la cosa (≈ 0,44-0,53) e passare a CBOW ancora di più (≈ 0,25). **Non esiste un «vicino corretto»**: c'è una lista che dipende da decisioni e dal caso. Con più testo (milioni di parole) la sovrapposizione tra semi sale; con qualche centinaio di migliaia di parole, come qui, è quello che vedi. Regola pratica: **prima di interpretare un vicino, verifica che compaia anche con un altro seme**.

In [ ]:
for idioma in FRASES:
    print("=" * 8, idioma, "| primi 6 vicini delle prime 3 ricerche, seme 1 contro seme 2")
    for palabra in CONSULTAS[idioma][:3]:
        print(f"{palabra:>10} s1:", vecinos(BASE[idioma], palabra), "| s2:", vecinos(OTROS[idioma]["seed 2"], palabra))

---
## 6. Analogie: una dimostrazione dei limiti

L'analogia famosa è `re − uomo + donna ≈ regina`. Con corpus enormi a volte funziona; con un corpus piccolo e antico **non c'è motivo di aspettarsi che funzioni**. La usiamo per misurare in quale **posizione** finisce la risposta attesa tra tutti i candidati (1 = la prima). Tre analogie fissate in anticipo per lingua.

In [ ]:
for idioma in FRASES:
    print("=" * 8, idioma)
    for a, b, c_, objetivo in ANALOGIAS[idioma]:
        posicion, primero = rango_analogia(BASE[idioma], a, b, c_, objetivo)
        if posicion is None:
            print(f"{a} - {b} + {c_} -> '{objetivo}' non è nel vocabolario")
        else:
            print(f"{a} - {b} + {c_} -> attesa '{objetivo}': posizione {posicion} de {len(BASE[idioma].wv) - 3} | primo candidato: '{primero}'")

### Cosa osservi?

I risultati sono **disomogenei**: `padre → madre` è prima in spagnolo, `señor → señora` quarta (la prima è `sobrina`), `rey → reina` è in posizione 61 su 10 143 (la prima è `hacienda`) e in italiano `re → regina` non si può nemmeno provare perché la parola attesa non è nel vocabolario. È quello che ci si aspetta con questa dimensione. Un'analogia che riesce in un esempio **non dimostra** che il modello «capisca» qualcosa: bisognerebbe provarne molte e contare i successi (con corpus molto più grandi si fa così, e il tasso di successo è lontano dal 100 %). Se in una demo vedi analogie perfette, sospetta che siano state scelte tra molte.

---
## 7. fastText: parole rare, fuori vocabolario e refusi

**fastText** (Bojanowski et al., 2017) rappresenta ogni parola come la **somma dei vettori dei suoi n-grammi di caratteri** (qui, da 3 a 5, con segni di inizio e fine: `<ca`, `cab`, `aba`…). Due conseguenze: (1) una parola **rara** condivide n-grammi con altre della stessa famiglia, quindi il suo vettore si appoggia su di esse; (2) una parola **che non è nel vocabolario** (un refuso, una forma derivata) **riceve comunque un vettore**, a partire dai suoi n-grammi. Word2Vec, invece, solleva un errore (`KeyError`). È molto utile nelle lingue a morfologia ricca (spagnolo, italiano: molte forme per verbo).

Scegliamo le parole rare **con una regola**, non a mano: le prime tre, in ordine alfabetico, con 5 o 6 occorrenze e almeno 8 lettere.

In [ ]:
FT = {idioma: entrenar_ft(FRASES[idioma], sg=1, window=5, seed=1) for idioma in FRASES}
for idioma in FRASES:
    print("=" * 8, idioma, "| vocabolario fastText:", len(FT[idioma].wv))
    cuenta = Counter(w for f in FRASES[idioma] for w in f)
    raras = sorted(w for w, n in cuenta.items() if n in (5, 6) and len(w) >= 8)[:3]
    for w in raras:
        print(f"{w:>14} ({cuenta[w]} volte) word2vec:", vecinos(BASE[idioma], w, 5), "| fastText:", vecinos(FT[idioma], w, 5))
    print("-" * 8, "parole fuori vocabolario e refusi")
    for w in FUERA_DE_VOCABULARIO[idioma]:
        en_w2v = w in BASE[idioma].wv.key_to_index
        print(f"{w:>14} nel vocabolario di word2vec? {en_w2v} | fastText:", vecinos(FT[idioma], w, 5))
    try:
        BASE[idioma].wv[FUERA_DE_VOCABULARIO[idioma][0]]
    except KeyError as error:
        print("word2vec ->", type(error).__name__)
    assert FT[idioma].wv[FUERA_DE_VOCABULARIO[idioma][0]].shape == (50,)

### Cosa osservi?

Sulle parole rare **word2vec dà vicini quasi casuali** (una parola vista 5 volte non ha quasi contesto) e **fastText dà vicini morfologici** (`abdicación` → `adulación`, `educación`, `abnegación`; `abbracciato` → `abbandonato`, `afferrato`): condividono la terminazione, non necessariamente il significato. Con i refusi, fastText azzecca quando il refuso conserva molti n-grammi (`cabalero` → `cabal`, `caballero`; `cavalliere` → `cavaliere`) e sbaglia quando no (`quijotesco` → `quijote` sì, ma anche `galeote`, `dote`). **È forma, non senso**: fastText non sa che `hidalguez` significhi qualcosa; vede solo pezzi simili.

---
## 8. Limiti: un vettore per parola e distorsioni del corpus

**Un vettore per parola.** `vino` è il verbo *venire* e anche la bevanda; `porta` (italiano) è l'uscio e il verbo *portare*. Il modello ha **un solo vettore** che mescola tutti gli usi, ponderati per la loro frequenza nel corpus.

**Distorsioni (*bias*).** Il modello riflette ciò che c'è nei testi. Chiediamo i vicini della direzione `donna − uomo` (e viceversa): escono le parole il cui contesto somiglia di più a quello dell'una che a quello dell'altro **in questi romanzi**, non una verità sul mondo.

In [ ]:
for idioma in FRASES:
    print("=" * 8, idioma)
    for w in POLISEMIA[idioma]:
        print(f"{w:>8} ({sum(1 for f in FRASES[idioma] for x in f if x == w)} volte) ->", vecinos(BASE[idioma], w, 8))
    a, b = GENERO[idioma]
    print(f"{a} - {b}:", [w for w, _ in BASE[idioma].wv.most_similar(positive=[a], negative=[b], topn=8)])
    print(f"{b} - {a}:", [w for w, _ in BASE[idioma].wv.most_similar(positive=[b], negative=[a], topn=8)])

### Cosa osservi?

Per `vino` e `porta` escono vicini di **un solo senso** (il verbo in `vino`: `llegando`, `partió`; l'uscio in `porta`: `soglia`, `finestra`): l'altro resta **cancellato**. `sale` (21 occorrenze) dà una lista senza senso chiaro: poche occorrenze, pochi contesti. Nelle direzioni di genere compaiono soprattutto **parentele, appellativi e aggettivi dei romanzi** (`madre`, `hijos`, `marquesa`, `señorita`, `sobrina`; `caballero`, `famoso`, `santo`) e qualche residuo senza senso (`ningún`, `ese`; in italiano `uomo − donna` dà `avvenire`, `oggetto`, `esercito`: quasi nessuno schema): è il ritratto di una letteratura dell'Ottocento e precedente, non un giudizio. Con embedding addestrati su testi di oggi e di internet le distorsioni sono altre, e restano: **un embedding eredita le distorsioni del suo corpus**.

---
## 9. Corpus letterario contro corpus sintetico moderno

Addestriamo skip-gram sulle recensioni e sui ticket del corso (circa 11 000 parole; `min_count=3` e 30 epoche perché resti un po' di vocabolario) e lo confrontiamo con il modello letterario. Misuriamo prima **quanto del vocabolario moderno conosce il modello letterario** (copertura di parole, non di tipi) e poi i vicini delle 5 prime parole frequenti del corpus moderno di almeno 6 lettere presenti in entrambi.

In [ ]:
for idioma in FRASES:
    mod = entrenar_w2v(MODERNO[idioma], sg=1, window=5, seed=1, epochs=30, min_count=3)
    cuenta = Counter(w for f in MODERNO[idioma] for w in f)
    total = sum(cuenta.values())
    cubiertas = sum(n for w, n in cuenta.items() if w in BASE[idioma].wv.key_to_index)
    ausentes = [w for w, n in cuenta.most_common(60) if w not in BASE[idioma].wv.key_to_index and len(w) >= 6][:6]
    comunes = [w for w, n in cuenta.most_common(300) if w in mod.wv.key_to_index and w in BASE[idioma].wv.key_to_index and len(w) >= 6][:5]
    print("=" * 8, idioma, "| vocabolario moderno:", len(mod.wv), "| copertura del vocabolario letterario sulle parole del testo moderno:", round(cubiertas / total, 2))
    print("Frequenti nel corpus moderno e assenti dal letterario:", ausentes)
    for w in comunes:
        print(f"{w:>12} letterario:", vecinos(BASE[idioma], w, 5), "| moderno:", vecinos(mod, w, 5))
    assert 0.5 < cubiertas / total < 0.95 and len(comunes) == 5

### Cosa osservi?

Le parole del dominio moderno (`paquete`, `devolución`, `factura`, `tarjeta`; `fattura`, `addebito`, `rimborso`) **non esistono** nel vocabolario letterario: un modello addestrato con Cervantes non può rappresentarle. La copertura globale sembra alta (0,81 in spagnolo e 0,77 in italiano) perché la fanno le parole grammaticali; **il 19-23 % che manca è soprattutto vocabolario del dominio**. Nelle parole comuni, i vicini letterari sono **verbi e arcaismi** (`necesito` → `pregunto`, `confieso`), quelli del corpus moderno sono di **pratiche** (`necesito` → `quisiera`, `extracto`): la stessa parola, due usi. Ma il modello moderno è addestrato su ~11 000 parole: i suoi vicini sono **poco affidabili** (e con un altro seme cambierebbero). Conclusione pratica: per un dominio concreto, o si addestra con **molto** testo di quel dominio, o si parte da vettori **preaddestrati** e si verifica che coprano il vocabolario.

---
## Riepilogo

- **Ipotesi distribuzionale:** il significato si approssima con il contesto. Gli embedding sono vettori **densi** appresi a partire dai contesti; il **coseno** confronta direzioni.
- **Word2Vec** (skip-gram/CBOW) e **fastText** si addestrano con `gensim`; la **finestra**, `min_count`, `vector_size`, le epoche e il **seme** cambiano il risultato.
- Con **qualche centinaio di migliaia di parole** i vicini sono **rumorosi**: verifica sempre con un altro seme. Le **analogie** sono una dimostrazione dei limiti, non un risultato.
- **fastText** dà un vettore a parole rare, fuori vocabolario e refusi, ma per **forma**, non per senso.
- Limiti: **un vettore per parola**, **distorsioni** ereditate dal corpus, copertura del dominio.

**Compito:** il notebook di pratica (`clase10_practica_word_embeddings_IT.ipynb`). **Prossima:** lezione 11, ricerca e recupero classico (TF-IDF, BM25): il recuperatore che gli embedding densi integrano.